In [35]:
import numpy as np
from scipy.special import exp1
import matplotlib.pyplot as plt

In [ ]:
def theis_u(r, T, S, t):
    """
    Calculate dimensionless Theis variable (u):
        r: Radial distance(s) from well(s) [L]
        T: Transmissivity [L^2/T]
        S: Storage coefficient [-]
        t: Time since pumping started [T]

    Notes:
        - r,t can be arrays (broadcastable).

    """
    r = np.asarray(r, dtype=float)
    t = np.asarray(t, dtype=float)
    if np.any(t <= 0):
        raise ValueError("t must be > 0 for Theis solution!")
    return (r**2 * S) / (4.0 * T * t)

In [37]:
def drawdown_s(Q, T, S, r, t):
    """
    Calculate drawdown:
        Q: pumping rate (positive for extraction) [L^3/T]
        T: transmissivity [L^2/T]
        S: storage coefficient [-]
        r: radial distance(s) [L] (scalar or array)
        t: time(s) since pumping started [T] (scalar or array)
    
    Notes:
        - r,t can be arrays (broadcastable).
    
    """
    u = theis_u(r, T, S, t)
    return (Q / (4.0 * np.pi * T)) * exp1(u)

In [38]:
def ds_dr_from_theis(Q, T, S, r, t):
    """
    Calculate radial derivative ds/dr from the Theis solution in a numerically stable form:
        Q: pumping rate (positive for extraction) [L^3/T]
        T: transmissivity [L^2/T]
        S: storage coefficient [-]
        r: radial distance(s) [L] (scalar or array)
        t: time(s) since pumping started [T] (scalar or array)

    Notes: 
        - ds/dr is singular at r = 0 (1/r)
        - For r = =0, function produces NaN

    """
    r = np.asarray(r, dtype=float)
    t = np.asarray(t, dtype=float)
    u = theis_u(r, T, S, t)
    out = - (Q / (2.0 * np.pi * T)) * (np.exp(-u) / r)

    # mark undefined at r==0
    if out.shape == ():
        if r == 0:
            return np.nan
    else:
        out = np.array(out, copy=True)
        out[r == 0] = np.nan
    return out

In [39]:
def radial_flux_qr(Q, T, S, r, t):
    """
    Calculate specific radial Darcy flux:
        Q: pumping rate (positive for extraction) [L^3/T]
        T: transmissivity [L^2/T]
        S: storage coefficient [-]
        r: radial distance(s) [L] (scalar or array)
        t: time(s) since pumping started [T] (scalar or array)
    
    Notes:
        - q_r has 1/r singularity at r = 0
    """
    r = np.asarray(r, dtype=float)
    u = theis_u(r, T, S, t)
    out = (Q / (2.0 * np.pi)) * (np.exp(-u) / r)
    if out.shape == ():
        if r == 0:
            return np.nan
    else:
        out = np.array(out, copy=True)
        out[r == 0] = np.nan
    return out

In [40]:
def boundary_flux(Q, T, S, r, t):
    """
    Calculate boundary flux:
        Q: pumping rate (positive for extraction) [L^3/T]
        T: transmissivity [L^2/T]
        S: storage coefficient [-]
        r: radial distance(s) [L] (scalar or array)
        t: time(s) since pumping started [T] (scalar or array)
    
    """
    u = theis_u(r, T, S, t)
    return Q * np.exp(-u)


In [44]:
# example parameters
Q = 1.0e-3      # m^3/s
T = 1e-3        # m^2/s
S = 1e-4        # dimensionless
t = 3600.0      # s

# radii from 1 m out to very large 1e4 m (10 km)
radii = np.array([1.0, 10.0, 100.0, 1000.0, 10000.0])

print("r (m)   u                s (m)              ds/dr (m/m)        q_r (m^2/s)         Q_boundary (m^3/s)")
for r in radii:
    u = theis_u(r, T, S, t)
    s = drawdown_s(Q, T, S, r, t)
    dsdr = ds_dr_from_theis(Q, T, S, r, t)
    qr = radial_flux_qr(Q, T, S, r, t)
    Qb = boundary_flux(Q, T, S, r, t)
    print(f"{r:7.1f}  {u:12.4e}  {s:14.6e}  {dsdr:14.6e}  {qr:14.6e}  {Qb:14.6e}")

r (m)   u                s (m)              ds/dr (m/m)        q_r (m^2/s)         Q_boundary (m^3/s)
    1.0    6.9444e-06    8.992541e-01   -1.591538e-01    1.591538e-04    9.999931e-04
   10.0    6.9444e-04    5.328410e-01   -1.590445e-02    1.590445e-05    9.993058e-04
  100.0    6.9444e-02    1.717496e-01   -1.484775e-03    1.484775e-06    9.329120e-04
 1000.0    6.9444e+00    9.784342e-06   -1.534215e-07    1.534215e-10    9.639757e-07
10000.0    6.9444e+02   2.918357e-306  -4.059102e-307   4.059102e-310   2.550409e-305
